# Chest X-Ray Classification using Xception

This notebook implements a chest X-ray classification model using the Xception architecture. It leverages Transfer Learning for high accuracy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
We use ImageDataGenerator for data augmentation and loading.

In [ ]:
cntr = 0
for i in ["train", "val", "test"]:
    for j in ["NORMAL", "PNEUMONIA"]:
        try:
            cntr += len(os.listdir(f'datasets/chest_xray/{i}/{j}'))
        except FileNotFoundError:
            print(f"Directory not found: datasets/chest_xray/{i}/{j}")
            
print('Total Images:', cntr)

img_size = 224
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    vertical_flip=True,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'datasets/chest_xray/train',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    'datasets/chest_xray/test',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = test_datagen.flow_from_directory(
    'datasets/chest_xray/val',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary'
)

## 2. Model Definition (Xception)
We use Xception as the base model, pre-trained on ImageNet, and add custom layers for our binary classification task.

In [ ]:
base_model = Xception(
    weights='imagenet',
    include_top=False,
    input_shape=(img_size, img_size, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Determine how many layers to freeze/unfreeze
# For fine-tuning, we might want to unfreeze some top layers
for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 3. Training & Evaluation

In [ ]:
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.4f}")

## 4. Save Model

In [ ]:
model.save('chest_xray_model.h5')
print("Model saved as chest_xray_model.h5")

## 5. Verification
Load the saved model and test on a random sample from the test set.

In [2]:
# Verification
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

# Load model
loaded_model = load_model('chest_xray_model.h5')

# Test on a sample batch (assuming test_generator is in memory)
try:
    x_test_batch, y_test_batch = next(test_generator)
    sample_image = x_test_batch[0]
    sample_label = y_test_batch[0]

    # Predict
    prediction = loaded_model.predict(np.expand_dims(sample_image, axis=0))
    print(f"Predicted Probability: {prediction[0][0]:.4f}")
    print(f"Actual Label: {sample_label}")

    # Show Image
    plt.imshow(sample_image)
    plt.title(f"Actual: {sample_label}, Pred: {prediction[0][0]:.4f}")
    plt.axis('off')
    plt.show()
except NameError:
    print("test_generator not found. Please run the notebook cells above to load data first.")

test_generator not found. Please run the notebook cells above to load data first.
